In [1]:
%pip install flask flask-cors


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
from flask import Flask, jsonify, request
from flask_cors import CORS
import sqlite3

app = Flask(__name__)
CORS(app)

WEATHER_OPTIONS = ["Hot", "Mild", "Snowy"]


def get_db_path():
    candidates = [
        Path.cwd() / "data" / "travel.db",
        Path.cwd() / ".." / "data" / "travel.db",
        Path.cwd().parent / "data" / "travel.db",
    ]

    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.exists():
            return resolved

    raise FileNotFoundError("Could not find data/travel.db from the current notebook folder.")


def get_db_connection():
    conn = sqlite3.connect(get_db_path())
    conn.row_factory = sqlite3.Row
    return conn


def weather_band_sql():
    return """
        CASE
            WHEN weather_monthly.avg_temp_c <= 5 THEN 'Snowy'
            WHEN weather_monthly.avg_temp_c >= 25 THEN 'Hot'
            ELSE 'Mild'
        END
    """


def normalize(value):
    return value.strip().lower() if isinstance(value, str) else ""


@app.route('/api/destinations', methods=['GET'])
def get_destinations():
    conn = get_db_connection()
    data = conn.execute("""
        SELECT
            destinations.id,
            destinations.name,
            destinations.destination_type,
            countries.name AS country,
            countries.region,
            countries.currency_code,
            countries.main_language
        FROM destinations
        JOIN countries ON countries.id = destinations.country_id
        ORDER BY destinations.name;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/activities', methods=['GET'])
def get_activities():
    conn = get_db_connection()
    data = conn.execute("""
        SELECT
            activities.id,
            destinations.name AS destination,
            activity_types.name AS activity_type,
            activities.activity_name
        FROM activities
        JOIN destinations ON destinations.id = activities.destination_id
        JOIN activity_types ON activity_types.id = activities.activity_type_id
        ORDER BY activity_types.name, destinations.name;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/vibes', methods=['GET'])
def get_vibes():
    conn = get_db_connection()
    data = conn.execute("SELECT * FROM travel_vibes ORDER BY name;").fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/filter-options', methods=['GET'])
def get_filter_options():
    conn = get_db_connection()
    cost_options = [row['budget_level'] for row in conn.execute("""
        SELECT DISTINCT budget_level
        FROM cost_profiles
        WHERE budget_level IS NOT NULL
        ORDER BY budget_level;
    """).fetchall()]
    activity_options = [row['name'] for row in conn.execute("""
        SELECT DISTINCT activity_types.name
        FROM activity_types
        JOIN activities ON activities.activity_type_id = activity_types.id
        ORDER BY activity_types.name;
    """).fetchall()]
    vibe_options = [row['name'] for row in conn.execute("""
        SELECT DISTINCT travel_vibes.name
        FROM travel_vibes
        JOIN destination_vibes ON destination_vibes.vibe_id = travel_vibes.id
        ORDER BY travel_vibes.name;
    """).fetchall()]
    conn.close()

    return jsonify({
        "cost": cost_options,
        "weather": WEATHER_OPTIONS,
        "activity": activity_options,
        "vibe": vibe_options,
    })


@app.route('/api/search', methods=['GET'])
def search_destinations():
    selected_filters = {
        "cost": request.args.get("cost", "").strip(),
        "weather": request.args.get("weather", "").strip(),
        "activity": request.args.get("activity", "").strip(),
        "vibe": request.args.get("vibe", "").strip(),
    }
    selected_filters = {key: value for key, value in selected_filters.items() if value}

    conn = get_db_connection()
    rows = conn.execute(f"""
        SELECT
            destinations.id AS destination_id,
            destinations.name AS destination,
            countries.name AS country,
            cost_profiles.budget_level AS cost,
            weather_monthly.avg_temp_c,
            weather_monthly.rainfall_mm,
            {weather_band_sql()} AS weather_band,
            activity_types.name AS activity_type,
            activities.activity_name,
            travel_vibes.name AS vibe
        FROM destinations
        JOIN countries ON countries.id = destinations.country_id
        JOIN cost_profiles ON cost_profiles.destination_id = destinations.id
        JOIN weather_monthly ON weather_monthly.destination_id = destinations.id
        JOIN activities ON activities.destination_id = destinations.id
        JOIN activity_types ON activity_types.id = activities.activity_type_id
        LEFT JOIN destination_vibes ON destination_vibes.destination_id = destinations.id
        LEFT JOIN travel_vibes ON travel_vibes.id = destination_vibes.vibe_id
        ORDER BY destinations.name, travel_vibes.name;
    """).fetchall()
    conn.close()

    destinations = {}
    for row in rows:
        destination_id = row['destination_id']
        if destination_id not in destinations:
            destinations[destination_id] = {
                "destination": row['destination'],
                "country": row['country'],
                "cost": row['cost'],
                "avg_temp_c": row['avg_temp_c'],
                "rainfall_mm": row['rainfall_mm'],
                "weather_band": row['weather_band'],
                "activity_type": row['activity_type'],
                "activity_name": row['activity_name'],
                "vibes": [],
            }

        if row['vibe'] and row['vibe'] not in destinations[destination_id]["vibes"]:
            destinations[destination_id]["vibes"].append(row['vibe'])

    results = []
    selected_filter_count = len(selected_filters)

    for destination in destinations.values():
        match_count = 0

        if normalize(selected_filters.get("cost", "")) == normalize(destination["cost"]):
            match_count += 1
        if normalize(selected_filters.get("weather", "")) == normalize(destination["weather_band"]):
            match_count += 1
        if normalize(selected_filters.get("activity", "")) == normalize(destination["activity_type"]):
            match_count += 1
        if selected_filters.get("vibe") and normalize(selected_filters["vibe"]) in [normalize(vibe) for vibe in destination["vibes"]]:
            match_count += 1

        destination["match_count"] = match_count
        destination["selected_filter_count"] = selected_filter_count
        results.append(destination)

    results.sort(key=lambda destination: (-destination["match_count"], destination["destination"]))
    return jsonify(results)


if __name__ == '__main__':
    app.run(port=5001, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [17/May/2026 21:58:19] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 21:58:29] "GET /api/search?cost=expensive&activity=architecture&vibe=Sun+%26+Sand HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 21:58:34] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 22:05:19] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 22:06:10] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 22:07:27] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 22:07:52] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 23:04:40] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 23:04:48] "GET /api/search HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 23:09:50] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 23:09:50] "GET /api/search HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 23:10:27] "GET /api/filter-